# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zajitario/flyrank-ML-repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Ranking signal analysis is a **ranking/scoring** problem. The decision I want to improve is: **which content pages should an editor or SEO analyst review first?** The output should not be a hard yes/no classification. It should be a prioritized list of pages, ordered by where the strongest combination of risk and opportunity appears.

A fixed rule is not enough because the pages that deserve attention are not defined by one universal threshold. The right priority depends on how several signals interact: traffic volume, freshness, search position, engagement, and the page's content type.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import io
import pandas as pd
from IPython.display import display

def load_starter_dataset():
    candidate_paths = [
        Path('../../data/raw/content_refresh_anonymized.csv'),
        Path('/content/data/raw/content_refresh_anonymized.csv'),
        Path('/content/content_refresh_anonymized.csv'),
    ]
    for path in candidate_paths:
        if path.exists():
            return pd.read_csv(path)

    fallback_url = 'https://raw.githubusercontent.com/Zajitario/flyrank-ML-repo/main/data/raw/content_refresh_anonymized.csv'
    with urlopen(fallback_url) as response:
        return pd.read_csv(io.BytesIO(response.read()))

df = load_starter_dataset()

print(f"Starter dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Distinct clients: {df['client_id'].nunique()}")
print("Relevant columns for ranking analysis:")
print([c for c in ['content_id', 'client_id', 'content_type', 'impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'trend_direction'] if c in df.columns])

Starter dataset shape: 30,000 rows x 44 columns
Distinct clients: 32
Relevant columns for ranking analysis:
['content_id', 'client_id', 'content_type', 'impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'trend_direction']


## 2. Target or proxy

For this lane, the best target is a **priority proxy**: a score that estimates which pages should be reviewed first. In the starter project, a related label already exists, `is_declining_label`, which is derived from `trend_direction == "down"`. That label is useful for learning the workflow, but it is still a proxy built from the current window, not the ideal long-term business target.

If I were building the lane for real, I would predict a ranking score that orders pages by review priority. The label should come from observed outcomes or a careful proxy defined from later behavior, not from a hand-written rule.

In [ ]:
# Sketch the kind of target/proxy shape this lane would use.
# This is intentionally simple and directional.

proxy = (
    df['impressions_90d'].rank(pct=True) * 0.4
    + (1 - df['avg_position'].fillna(df['avg_position'].max()).rank(pct=True)) * 0.3
    + df['days_since_last_update'].rank(pct=True) * 0.3
).round(3)

preview = df[['content_id', 'client_id', 'impressions_90d', 'avg_position', 'days_since_last_update', 'trend_direction']].head(8).copy()
preview['priority_proxy'] = proxy.head(8).values

display(preview)

,content_id,client_id,impressions_90d,avg_position,days_since_last_update,trend_direction,priority_proxy
0,content_304f48230142,client_f369cb89fc,3803,10.6,20,down,0.555
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,25,down,0.649
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,20,down,0.493
3,content_331d6c4de07b,client_19581e27de,11751,6.2,22,stable,0.761
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,14,down,0.431
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,20,down,0.585
6,content_9a34b442b552,client_8722616204,20,7.0,20,down,0.374
7,content_a63219c6e95a,client_19581e27de,1724,21.2,22,stable,0.511


## 3. Success metric

A defensible success metric for a ranking lane is **precision@K**. In practice, that means asking whether the top K pages returned by the score are truly the ones an analyst would want to review first.

For this notebook, I would use **precision@20** as the first-pass metric. It is easy to explain, it matches a small human review queue, and it rewards putting the most useful pages at the top of the list.

In [ ]:
# Precision@K is a ranking metric, so this cell shows the top-K queue that the metric would evaluate.

k = 20
ranked_queue = (
    df[['content_id', 'client_id']]
    .assign(priority_proxy=proxy)
    .sort_values('priority_proxy', ascending=False)
    .head(k)
    .reset_index(drop=True)
)

print(f'If I review the top {k} pages, precision@{k} would measure how many of them are truly worth that attention.')
display(ranked_queue)

If I review the top 20 pages, precision@20 would measure how many of them are truly worth that attention.


,content_id,client_id,priority_proxy
0,content_9532f197bbc8,client_4e07408562,0.937
1,content_69fad7e6c50c,client_7f2253d7e2,0.937
2,content_03d2673b2553,client_19581e27de,0.936
3,content_4a6607efcb46,client_6208ef0f77,0.934
4,content_6ac3ab740bbf,client_f369cb89fc,0.934
5,content_654d006adc44,client_19581e27de,0.933
6,content_7a6df559322d,client_19581e27de,0.932
7,content_4d1fe5b32dc2,client_19581e27de,0.931
8,content_9c195417f6ef,client_19581e27de,0.930
9,content_9351f948bf45,client_19581e27de,0.930


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# Show the actual page-level dataframe used for framing the lane.

unit_of_analysis = df[[
    'content_id',
    'client_id',
    'content_type',
    'content_age_days',
    'days_since_last_update',
    'impressions_90d',
    'sessions_90d',
    'ctr',
    'avg_position',
    'trend_direction',
]].copy()

display(unit_of_analysis.head(10))
print(f'Unit of analysis: one row = one content item / page')
print(f'Slice shape: {unit_of_analysis.shape[0]:,} rows x {unit_of_analysis.shape[1]} columns')

,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,sessions_90d,ctr,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,17,0.76,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,9,0.05,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,0.09,36.5,down
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,78,0.49,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,145,0.13,44.0,down
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,5,0.03,8.5,down
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,1,0.00,7.0,down
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,28,0.06,21.2,stable
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,68,0.09,46.0,down
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,3,0.16,4.9,down


Unit of analysis: one row = one content item / page
Slice shape: 30,000 rows x 10 columns


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# A fixed rule would need a long list of brittle thresholds.
# This cell shows how many different signal families are present in the starter data.

signal_groups = {
    'traffic': ['impressions_90d', 'clicks_90d', 'sessions_90d'],
    'quality': ['ctr', 'avg_position', 'engagement_rate', 'scroll_rate'],
    'freshness': ['content_age_days', 'days_since_last_update'],
    'context': ['content_type', 'main_intent', 'competition_level'],
}

for group_name, columns in signal_groups.items():
    available = [column for column in columns if column in df.columns]
    print(f"{group_name}: {available}")

print("ML is useful here because priority depends on combinations of signals, not one universal cutoff.")

traffic: ['impressions_90d', 'clicks_90d', 'sessions_90d']
quality: ['ctr', 'avg_position', 'engagement_rate', 'scroll_rate']
freshness: ['content_age_days', 'days_since_last_update']
context: ['content_type', 'main_intent', 'competition_level']
ML is useful here because priority depends on combinations of signals, not one universal cutoff.


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.